In [0]:
WITH CA_DIGITAL_WORKSPACE AS (
SELECT  `Parent._id`,
        --ROW_NUMBER() OVER (PARTITION BY `Parent._id` ORDER BY `Parent._id` DESC) AS Rn, 
        Created_by,
        Start_ActedAt,
        Type_of_request,
        TRIM(CASE WHEN regexp_count(Customer_Selected, '[0-9]') > 4 THEN regexp_replace(Customer_Selected, '[-/|0-9]', '') ELSE Customer_Selected END) AS Customer_Selected,  
        CASE WHEN Account_1 = '' THEN 
             CASE WHEN regexp_count(Customer_Selected, '[0-9]') > 4 THEN  
                  array_join(regexp_extract_all(Customer_Selected, '[0-9]', 0), '') 
                  ELSE Account_1 
             END
             ELSE Account_1
        END AS Account_1,
        CASE WHEN regexp_replace(split_part(split_part(Account_Details, ':', 3),',', 1), '["]', '') = '' THEN Customer_Selected ELSE regexp_replace(split_part(split_part(Account_Details, ':', 3),',', 1), '["]', '') END AS Customer_Name,
        -- SE ELIMINAN DIGITOS Y CARACTERES ESPECIALES DE COLUMNA Customer_Name EN NUEVA COLUMNA Customer_Name_
        TRIM(CASE WHEN regexp_count(Customer_Name, '[0-9]') > 4 THEN regexp_replace(Customer_Name, '[-/|0-9]', '') ELSE Customer_Name END) AS Customer_Name_,
        -- SE EXTRAEN CÓDIGOS CLIENTE Y SE REEMPLAZAN NULLS DE Customer_Number POR Import_Customer_Number SINO POR New_Customer_Account_Details.
        CASE WHEN regexp_replace(split_part(split_part(Account_Details, ':', -3), ',', 1), '[-"A-Z]', '') = '' THEN
             CASE WHEN Import_Customer_Number = '' THEN 
                  CASE WHEN New_Customer_Account_Details = '' THEN 
                       regexp_replace(split_part(split_part(Account_Details, ':', -3), ',', 1), '[-"A-Z]', '')
                       WHEN New_Customer_Account_Details <> '' THEN
                       CASE WHEN regexp_count(New_Customer_Account_Details, '[0-9]') > 4 THEN  
                            array_join(regexp_extract_all(New_Customer_Account_Details, '[0-9]', 0), '')
                            ELSE regexp_replace(split_part(split_part(Account_Details, ':', -3), ',', 1), '[-"A-Z]', '')
                       END
                  END 
                  ELSE Import_Customer_Number
             END 
             ELSE regexp_replace(split_part(split_part(Account_Details, ':', -3), ',', 1), '[-"A-Z]', '')
        END AS Customer_Number,
        New_Customer_Account_Details,  
        Multiple_Account_Details,
        Import_Customer_Number,
        TRIM(CASE WHEN regexp_count(Group_Name, '[0-9]') > 4 THEN regexp_replace(Group_Name, '[-/|0-9]', '') ELSE Group_Name END) AS Group_Name,
        CASE WHEN CountryEntity = '' THEN regexp_replace(split_part(split_part(Account_Details, ':', -5), ',', 1), '["]', '') ELSE CountryEntity END AS CountryEntity,
        Region,
        CAST(Current_Credit_Limit AS DOUBLE) AS Current_Credit_Limit_USD, 
        CAST(Current_Exposure_USD AS DOUBLE),
        CAST(Current_Payment_Terms_Days AS DOUBLE),
        CAST(Recommended_Credit_Limit AS DOUBLE),
        CAST(Requested_Credit_Coverage_USD AS DOUBLE),
        CAST(Requested_Payment_Terms_Days AS DOUBLE),
        CAST(Security_Offered_USD AS DOUBLE),
        CAST(EconomicsGross_Margin_USD AS DOUBLE),
        Status, 
        Modified_by,
        Modified_at,
        Current_Step,
        array(
          regexp_replace(split_part(split_part(Current_Assigned_to, ',', 2), ':', 2), '["]', ''),   -- SE OBTIENEN LOS NOMBRES DE CADA UNO DE LOS APROBADORES ASIGNADOS EN EL 
          regexp_replace(split_part(split_part(Current_Assigned_to, ',', 5), ':', 2), '["]', ''),    -- MOMENTO, Y SE ALMACENAN DENTRO DE UNA AGRUPACIÓN (ARRAY)
          regexp_replace(split_part(split_part(Current_Assigned_to, ',', 8), ':', 2), '["]', ''),
          regexp_replace(split_part(split_part(Current_Assigned_to, ',', 11), ':', 2), '["]', '') 
        ) AS Current_Assigned_to,
        FM_approval_ActedAt,
        regexp_replace(split_part(split_part(FM_approval_ActedBy, ',', 2), ':', 2), '["]', '') AS FM_approval_ActedBy, 
        Cluster_Finance_Manager_Cluster_Credit_Manager_ActedAt,
        regexp_replace(split_part(split_part(Cluster_Finance_Manager_Cluster_Credit_Manager_ActedBy, ',', 2), ':', 2), '["]', '') AS Cluster_Finance_Manager_Cluster_Credit_Manager_ActedBy,
        Regional_Cluster_Credit_Manager_Approval_ActedAt,
        regexp_replace(split_part(split_part(Regional_Cluster_Credit_Manager_Approval_ActedBy, ',', 2), ':', 2), '["]', '') AS Regional_Cluster_Credit_Manager_Approval_ActedBy,
        Region_Head_Approval_ActedAt,
        regexp_replace(split_part(split_part(Region_Head_Approval_ActedBy, ',', 2), ':', 2), '["]', '') AS Region_Head_Approval_ActedBy, 
        RFM_Approval_ActedAt,
        regexp_replace(split_part(split_part(RFM_Approval_ActedBy, ',', 2), ':', 2), '["]', '') AS RFM_Approval_ActedBy, 
        Chairman_Approval_ActedAt,
        regexp_replace(split_part(split_part(Chairman_Approval_ActedBy, ',', 2), ':', 2), '["]', '') AS Chairman_Approval_ActedBy, 
        Regional_Finance_Manager_Region_Head_Regional_COO_Chairman_LATAM_Bitumen_Head_ActedAt,
        regexp_replace(split_part(split_part(Regional_Finance_Manager_Region_Head_Regional_COO_Chairman_LATAM_Bitumen_Head_ActedBy, ',', 2), ':', 2), '["]', '') AS Regional_Finance_Manager_Region_Head_Regional_COO_Chairman_LATAM_Bitumen_Head_ActedBy,
        Regional_CFO_ActedAt,
        regexp_replace(split_part(split_part(Regional_CFO_ActedBy, ',', 2), ':', 2), '["]', '') AS Regional_CFO_ActedBy, 
        Regional_CFO_Head_of_Trade_Finance_Credit_ActedAt,
        regexp_replace(split_part(split_part(Regional_CFO_Head_of_Trade_Finance_Credit_ActedBy, ',', 2), ':', 2), '["]', '') AS Regional_CFO_Head_of_Trade_Finance_Credit_ActedBy 
        --CFO_Approval_ActedAt,
        --regexp_replace(split_part(split_part(CFO_Approval_ActedBy, ',', 2), ':', 2), '["]', '') AS CFO_Approval_ActedBy
        --CEO_Board_Approval_ActedAt,
        --regexp_replace(split_part(split_part(CEO_Board_Approval_ActedBy, ',', 2), ':', 2), '["]', '') AS CEO_Board_Approval_ActedBy

FROM kissflow.default.credit_assessment_digital_workspace_l
WHERE CountryEntity <> 'Aviation' 
AND Created_by != 'Ekta Ramparia'
AND Created_by != 'jesus fiallos'
AND Created_by NOT LIKE 'Brayan%'
AND Created_at >= '2025-01-01'
),

CA_DIGITAL_WORKSPACE_GROUPBY_PARENTID AS (            -- SE AGRUPA POR `parent._id` PARA IDENTIFICAR SI DENTRO DE DICHO ASSESSMENT SE INCLUYEN VARIOS CÓDIGOS. 
SELECT `parent._id`, count(*) AS Count_Parentid
FROM CA_DIGITAL_WORKSPACE
GROUP BY `parent._id`
),

CA_DIGITAL_WORKSPACE_ (
SELECT * EXCEPT(Customer_Name, Customer_Number, new_customer_account_details, 
                multiple_account_details, 
                import_customer_number, 
                CA_G.`parent._id`), 
       CASE WHEN CA.Customer_Number = '' THEN 
            CASE WHEN CA_G.Count_Parentid = 1 AND CA.Account_1 <> '' THEN  -- NULLs VALUES QUE AÚN PERMANECEN EN LA COLUMNA Customer_Number, SE REEMPLAZAN POR Account_1, 
                 CA.Account_1                                                -- SIEMPRE Y CUANDO EL parent._id SEA ÚNICO (Count_Parentid = 1), ES DECIR, QUE DICHO ASSESSMENT  
                 ELSE CA.Customer_Number                                      -- SOLO CUENTA CON UN REGISTRO/CÓDIGO.
            END 
            ELSE CA.Customer_Number
       END AS Customer_Number
FROM CA_DIGITAL_WORKSPACE CA
LEFT JOIN CA_DIGITAL_WORKSPACE_GROUPBY_PARENTID CA_G
 ON CA.`parent._id` = CA_G.`parent._id`
)

SELECT DW.*,
       CA.Key_Money_Amount,  
       CA.Financial_Strength, 
       CA.Select_Security_Type,  
       CA.`_progress`,
       CA.Flow_ID_for_Link
       
FROM CA_DIGITAL_WORKSPACE_ DW
LEFT JOIN kissflow.default.databricks_output_2025_08_29 CA 
 ON DW.`parent._id` = CA.`_id`
--AND DW.start_actedat = CA.`_submitted_at.display_value`
--WHERE regexp_count(DW.new_customer_account_details, '[0-9]') > 4
--AND DW.count_parentid <> 1
--AND DW.rn = 1 AND DW.account_1 <> ''
--AND DW.rn_cust_number > 1
ORDER BY DW.start_actedat DESC;


-- PENDIENTE: 
          -- VALIDAR SI ES POSIBLE RECUPERAR LOS NULL VALUES DE Customer_Number RESTANTES CON COLUMNA Account_1; PARA EL RESTO QUE AÚN PERMANEZCAN NULL, VALIDAR UN CRUCE CON OTRA TABLA EXTERNA AL CATÁLOGO DE KISSFLOW. 

--CAMPOS NULOS: 
          --CUSTOMER_SELECTED: REGISTROS NULOS NO SE PUEDEN COMPLETAR YA QUE DEMÁS COLUMNAS NO TRAEN INFO. QUE SE PUEDA RELACIONAR;VALIDAR UN CRUCE CON OTRA TABLA EXTERNA AL CATÁLOGO DE KISSFLOW.
          --ACCOUNT_1: REGISTROS NULOS NO SE PUEDEN COMPLETAR YA QUE DEMÁS COLUMNAS NO TRAEN INFO. QUE SE PUEDA RELACIONAR;VALIDAR UN CRUCE CON OTRA TABLA EXTERNA AL CATÁLOGO DE KISSFLOW.
          --CUSTOMER_NAME_: REGISTROS NULOS NO SE PUEDEN COMPLETAR YA QUE DEMÁS COLUMNAS NO TRAEN INFO. QUE SE PUEDA RELACIONAR; VALIDAR UN CRUCE CON OTRA TABLA EXTERNA AL CATÁLOGO DE KISSFLOW.

